# 剔除解释层位异常值@20260410

本笔记本检查在`interpolate_interpretation_surface()`函数内部的孤立异常值步骤中，移除了多少解释点，针对以下数据：

- `data/interpre_time/bve_top_t`

- `data/interpre_time/itp_bot_t`

移除点的数量从`out_df.attrs["outlier_removal"]`中读取。


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cup.petrel.load import import_interpretation_petrel
from cup.seismic.process import _build_line_axis, interpolate_interpretation_surface
from cup.seismic.survey import open_survey
from ginn.config import GINNConfig

pd.set_option("display.max_columns", None)

In [ ]:
cfg = GINNConfig()

OUTLIER_THRESHOLD = 0.01
MIN_NEIGHBOR_COUNT = 2

horizon_files = {
    "top": PROJECT_ROOT / cfg.top_horizon_file,
    "bottom": PROJECT_ROOT / cfg.bot_horizon_file,
}

horizon_files

Build the same time-domain geometry used by `ginn.data.build_dataset()`.


In [ ]:
seismic_ctx = open_survey(
    PROJECT_ROOT / cfg.seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": cfg.segy_iline,
        "xline": cfg.segy_xline,
        "istep": cfg.segy_istep,
        "xstep": cfg.segy_xstep,
    },
)

geometry = seismic_ctx.query_geometry(domain="time")
il_axis = _build_line_axis(
    int(geometry["inline_min"]),
    int(geometry["inline_max"]),
    int(geometry["inline_step"]),
)
xl_axis = _build_line_axis(
    int(geometry["xline_min"]),
    int(geometry["xline_max"]),
    int(geometry["xline_step"]),
)

geometry

In [ ]:
def summarize_horizon(horizon_name: str, horizon_file: Path) -> dict:
    raw_df = import_interpretation_petrel(horizon_file)
    out_df = interpolate_interpretation_surface(
        interpretation_df=raw_df,
        geometry=geometry,
        outlier_threshold=OUTLIER_THRESHOLD,
        min_neighbor_count=MIN_NEIGHBOR_COUNT,
        keep_nan=True,
    )

    stats = dict(out_df.attrs["outlier_removal"])
    stats.update(
        {
            "horizon": horizon_name,
            "file": str(horizon_file.relative_to(PROJECT_ROOT)),
            "raw_rows": int(len(raw_df)),
            "output_rows": int(len(out_df)),
            "output_valid_points": int(out_df["interpretation"].notna().sum()),
        }
    )
    return stats

In [ ]:
rows = [summarize_horizon(horizon_name, horizon_file) for horizon_name, horizon_file in horizon_files.items()]

summary = pd.DataFrame(rows)
summary

Show horizons where the despike step removed at least one gridded interpretation point.


In [ ]:
summary.loc[summary["removed_points"] > 0]